In [35]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,roc_auc_score, confusion_matrix, roc_curve
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import random
def set_seed(seed=42):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)
set_seed(42)
scale=StandardScaler()

Grid Search of LSTM using no window dataframe results even though the preliminary results show that the window version is better. The grisearch performs better on non window (.58 f1 score) as opposed to .56 in windowed

In [36]:
#import dataframe
df_modelling=pd.read_parquet("../df_modelling.parquet")

In [37]:
#creating market vector
# 'Stock_Date', 'Closing Price($)', 'relative difference',
#        'comment_published_at_median_YMD', 'number_of_comments_at_date',
#        'comment_likes_agg', 'comment_embeddings_agg', 'comment_sentiment_agg',
#        'sentiment_tensor', 'videos_text_embeddings', 'TargetVariable:Up/Down'
numerical_columns=['Closing Price($) lag1',
'relative difference lag1',
         'number_of_comments_at_date',
    'comment_likes_agg',
    'comment_sentiment_agg',
     'sentiment_tensor']

print(df_modelling.columns)


#concating the embeddings separately
df_modelling['embeddings_vector']=df_modelling.apply(lambda row: np.concatenate((row['comment_embeddings_agg'], row['videos_text_embeddings'])), axis=1)
#concatenating the nummerical columns after that
#df_modelling['marketvector']=df_modelling.apply(lambda row: np.concatenate((row[numerical_columns].to_numpy(dtype=float),row['marketvector'])), axis=1)


Index(['Stock_Date', 'Closing Price($)', 'relative difference',
       'comment_published_at_YMD', 'number_of_comments_at_date',
       'comment_likes_agg', 'comment_embeddings_agg', 'comment_sentiment_agg',
       'sentiment_tensor', 'videos_text_embeddings',
       'videos_text_embeddings_window', 'TargetVariable:Up/Down',
       'Closing Price($) lag1', 'relative difference lag1'],
      dtype='object')


In [38]:
#function for creating sequences
def create_sequences(embeddings, numeric, target, sequence_length):
    X_emb_seq = []
    X_num_seq = []
    y_seq = []

    for i in range(sequence_length, len(embeddings)):
        X_emb_seq.append(embeddings[i-sequence_length:i])
        X_num_seq.append(numeric[i-sequence_length:i])
        y_seq.append(target[i])

    return (
        np.array(X_emb_seq),
        np.array(X_num_seq),
        np.array(y_seq)
    )

In [39]:
#splitting data into embeddings and numerical
embeddings_array = np.stack(df_modelling['embeddings_vector'])
numerical_values = np.stack(df_modelling[numerical_columns].values)
y = df_modelling['TargetVariable:Up/Down'].values



# # Raw splits
# emb_train_raw = embeddings_array[:train_end]
# emb_val_raw = embeddings_array[train_end:val_end]
# emb_test_raw = embeddings_array[val_end:]

# num_train_raw = numerical_values[:train_end]
# num_val_raw = numerical_values[train_end:val_end]
# num_test_raw = numerical_values[val_end:]

# y_train_raw = y[:train_end]
# y_val_raw = y[train_end:val_end]
# y_test_raw = y[val_end:]



In [40]:
def grid_LSTM(sequence_length,units,secondary_units, dropout, lr):
    
    #LSTM will look 20 steps in the past
    X_emb, X_num, y_seq = create_sequences(embeddings_array, numerical_values, y, sequence_length)
    # Splitting 70 train ,20 val and 10 testN = len(X_emb)
    N=len(X_emb)
    train_end = int(N * 0.7)
    val_end = int(N * 0.9)

    X_emb_train = X_emb[:train_end]
    X_emb_val   = X_emb[train_end:val_end]
    X_emb_test  = X_emb[val_end:]

    # Numerical
    X_num_train = X_num[:train_end]
    X_num_val   = X_num[train_end:val_end]
    X_num_test  = X_num[val_end:]

    # Targets 
    y_train = y_seq[:train_end]
    y_val   = y_seq[train_end:val_end]
    y_test  = y_seq[val_end:]
    #scaling just the numericals
    # After splitting sequences

    X_num_train = scale.fit_transform(
        X_num_train.reshape(-1, X_num_train.shape[-1])
    ).reshape(X_num_train.shape)

    X_num_val = scale.transform(
        X_num_val.reshape(-1, X_num_val.shape[-1])
    ).reshape(X_num_val.shape)

    X_num_test = scale.transform(
        X_num_test.reshape(-1, X_num_test.shape[-1])
    ).reshape(X_num_test.shape)

    # Inputs
    embedding_input = Input(shape=(sequence_length,X_emb_train.shape[2]))  
    numeric_input = Input(shape=(sequence_length,X_num_train.shape[2]))

    # Embeddings branch
    x = LSTM(units=units, return_sequences=True, dropout=dropout)(embedding_input)
    x = LSTM(secondary_units, dropout=dropout)(x)

    #numerical branch
    z= LSTM(units=units//2, return_sequences=True, dropout=dropout)(numeric_input)
    z= LSTM(units=secondary_units//2, dropout=dropout)(z)
    # Concatenate numeric features
    x = Concatenate()([x, z])

    # Output
    output = Dense(1, activation='sigmoid')(x)

    # Build model
    model = Model(inputs=[embedding_input, numeric_input], outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    model.summary()


    print(X_emb_train.shape)
    #early stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    history = model.fit(
        [X_emb_train, X_num_train],  # inputs
        y_train,                        # labels
        validation_data=([X_emb_val, X_num_val], y_val),
        epochs=200,                      # start with 10, can increase
        batch_size=32,                  # tune depending on memory
        shuffle=False, callbacks=[early_stop]
    )

    thresholds=np.arange(0.0,1.01,0.1)
    f1_scores=[]
    #GETTING THRESHOLD FROM VAlIDATION
    
    y_val_prob = model.predict([X_emb_val,X_num_val]).flatten()
    for threshold in thresholds:
        y_prob_tr = (y_val_prob > threshold).astype(int)
        report=classification_report(y_val, y_prob_tr, output_dict=True)
        f1=report['macro avg']['f1-score']
        f1_scores.append(f1)
    #saving the best threshold
    f1_score=max(f1_scores)
    index_tr=f1_scores.index(f1_score)
    threshold=thresholds[index_tr]
    y_test_prob = model.predict([X_emb_test, X_num_test]).flatten()
    y_prob_tr = (y_test_prob > threshold).astype(int)
    #testing on test
    report=classification_report(y_test, y_prob_tr, output_dict=True)
    best_cm=confusion_matrix(y_test,y_prob_tr.astype(int))
    f1=report['macro avg']['f1-score']
    
    return f1,threshold, report, best_cm
    
        

In [41]:
results=[]
param_grid = {
    "timesteps": [20,50],                   # timesteps
    "units": [128,64],                      # primary hidden units 
    "second_units":[32,16],                 # secondary hidden units              
    "dropout": [0.2, 0.5],                  # dropout
    "learning_rate": [1e-3,5e-4],           # learning rate
}                               
# grid search
for t in param_grid['timesteps']:
    for units in param_grid['units']:
        for s_unit in param_grid['second_units']:
            for dropout in param_grid['dropout']:
                for lr in param_grid["learning_rate"]:
                    f1_score,threshold, best_report, cm=grid_LSTM(t, units,s_unit, dropout, lr)
                    results.append({
                        "timesteps": t,
                        "units": units,
                        "second units":s_unit,
                        "dropout": dropout,
                        "learning_rate": lr,
                        "f1_score": f1_score,
                        "threshold":threshold,
                        "best_report":best_report, 
                        "confusion_matrix":cm
                    })
        


Model: "functional_96"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_194     │ (None, 20, 512)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_195     │ (None, 20, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_323 (LSTM)     │ (None, 20, 128)   │    328,192 │ input_layer_194[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_325 (LSTM)     │ (None, 20, 64)    │     18,176 │ input_layer_195[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_324 (LSTM)     │ (None, 32)        │     20,608 │ lstm_323[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_326 (LSTM)     │ (None, 16)        │      5,184 │ lstm_325[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_97      │ (None, 48)        │          0 │ lstm_324[0][0],   │
│ (Concatenate)       │                   │            │ lstm_326[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_96 (Dense)    │ (None, 1)         │         49 │ concatenate_97[0… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 372,209 (1.42 MB)

 Trainable params: 372,209 (1.42 MB)

 Non-trainable params: 0 (0.00 B)

...

/opt/anaconda3/lib/python3.12/site-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor_957', 'keras_tensor_958']. Received: the structure of inputs=('*', '*')
  warnings.warn(


10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 190ms/step - accuracy: 0.4699 - loss: 0.6998 - val_accuracy: 0.4545 - val_loss: 0.7056
Epoch 2/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 80ms/step - accuracy: 0.5395 - loss: 0.6938 - val_accuracy: 0.4545 - val_loss: 0.7056
Epoch 3/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.5351 - loss: 0.6875 - val_accuracy: 0.4545 - val_loss: 0.7070
Epoch 4/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - accuracy: 0.5174 - loss: 0.6921 - val_accuracy: 0.4545 - val_loss: 0.7083
Epoch 5/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.5516 - loss: 0.6898 - val_accuracy: 0.4545 - val_loss: 0.7082
Epoch 6/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 74ms/step - accuracy: 0.5481 - loss: 0.6883 - val_accuracy: 0.4545 - val_loss: 0.7083
Epoch 7/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - accuracy: 0.5204 - loss: 0.6933 - val_accuracy: 0.4545 - val_loss: 0.7083
Epoch 8/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - accuracy: 0.4967 - loss: 0.6929 - val_accuracy: 0.4545 - val_

/Users/irtg/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/irtg/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/irtg/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/irtg/.

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step


/Users/irtg/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/irtg/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/irtg/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [42]:
# auc=roc_auc_score(y_test, y_prob)

# fpr, tpr, thresholds = roc_curve(y_test, y_prob)

# plt.figure(figsize=(6,6))
# plt.plot(fpr, tpr, label=f'ROC curve (AUC = {auc:.2f})')
# plt.plot([0, 1], [0, 1], 'k--')  # random line
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('ROC Curve')
# plt.legend(loc='lower right')
# plt.title("ROC for LSTM with 512 embeddings and 6 numerical features")
# plt.show()

In [43]:
pd_results=pd.DataFrame(results)
pd_results.to_csv("GridSearchLSTM_without_window.csv")